<div style="padding:20px; border:1px solid #d0d7de; border-radius:12px; background:#f8fafc;">

<div align="center">

# CMAscan: Detection and Scoring of CMA-Targeting Motifs

</div>

CMAscan scans multi-FASTA protein sequences and reports canonical, phospho-dependent, and acetyl-dependent CMA motif candidates with integrated **cPSSM**, **ePSSM**, and optional **IUPred3** scoring.

**Runtime requirements**
- Internet access is required to download scoring resources.
- Large input files (hundreds of proteins) may take 10–30 minutes. Do not
   close the browser tab while the pipeline is running.
- Output column `localization` is reported as **1-based motif start position** in the input protein sequence.
- IUPred3 disorder scoring is optional and disabled by default. To enable,
  set `USE_IUPRED3 = True` in Configuration. In Step 1, you will be prompted
  to upload `iupred3.tar.gz` (download from https://iupred3.elte.hu/ under
  your own academic license).

**Quick Start**
1. Choose configuration.

Configuration guide:
- **Use demo input = ON**: run CMAscan on a built-in example dataset (best for a first run, no FASTA upload required).
- **Use demo input = OFF**: analyze your own proteins; Step 1 will ask you to upload a FASTA file.
- **Use IUPred3 = ON**: enable disorder analysis for each motif; Step 1 will ask you to upload `iupred3.tar.gz`.
- **Pseudophospho residue (E/D)**: choose which substitution should mimic phosphorylation in pseudo-PTM rescoring.

</div>

In [ ]:
#@title Configuration
# Run with pre-built example (no upload needed — recommended for first run)
USE_DEMO_INPUT = False  #@param {type:"boolean"}
# Enable IUPred3 disorder scoring (requires iupred3.tar.gz from https://iupred3.elte.hu/)
USE_IUPRED3 = False  #@param {type:"boolean"}
# Residue used to mimic phosphorylated S/T/Y. Default 'E' recommended.
PSEUDOPHOSPHO_RESIDUE = 'E'  #@param ['E', 'D']
# Name of output CSV file downloaded at the end
OUTPUT_FILE_NAME = 'motif_hits_scored.csv'  #@param {type:"string"}


<div style="padding:20px; border:1px solid #d0d7de; border-radius:12px; background:#f8fafc;">

2. Click **Runtime → Run all**.
3. Results appear in Step 12 and download automatically.

**Troubleshooting**
If the pipeline stops, shows an error, or the "Choose Files" button does not
appear:
1. Click **Runtime → Disconnect and delete runtime**.
2. Click **Runtime → Run all** to restart from scratch.

Do not run individual cells manually — the pipeline requires all previous
steps to be executed in order.

**External tools**
This notebook can optionally use IUPred3 (https://iupred3.elte.hu/) for
disorder scoring when `USE_IUPRED3 = True`. If you publish results obtained
with CMAscan + IUPred3, please also cite the IUPred3 authors — see **Step 8**
for full references.

</div>

In [ ]:
#@title Step 1: Upload or Select Input File

import tempfile as _tempfile
from google.colab import files as _files

if USE_DEMO_INPUT:
    import urllib.request as _ur
    _DEMO_URL = ('https://raw.githubusercontent.com/'
                 'Marcin-Lubocki/CMAscan/main/dataset/demo_input.fasta')
    INPUT_FILE_PATH = '/content/demo_input.fasta'
    print('Downloading demo input...')
    _ur.urlretrieve(_DEMO_URL, INPUT_FILE_PATH)
    print(f'Demo input loaded: {INPUT_FILE_PATH}')
else:
    print('Upload your FASTA file (.fasta, .faa, or .txt):')
    _uploaded = _files.upload()
    if not _uploaded:
        print('=' * 60)
        print('  ⚠  CMAscan: Invalid FASTA file')
        print('=' * 60)
        print()
        print('No file was uploaded.')
        print()
        print('Please correct your file and run the pipeline again.')
        print('Reload via Step 1 (Choose Files) after fixing.')
        print('=' * 60)
        raise SystemExit(1)
    _fname = list(_uploaded.keys())[0]
    INPUT_FILE_PATH = f'/content/{_fname}'
    with open(INPUT_FILE_PATH, 'wb') as _f:
        _f.write(_uploaded[_fname])
    print(f'Input file saved: {INPUT_FILE_PATH}')

if USE_IUPRED3:
    IUPRED3_ARCHIVE_PATH = '/content/iupred3.tar.gz'
    from pathlib import Path as _P
    if not _P(IUPRED3_ARCHIVE_PATH).is_file():
        print()
        print('Upload iupred3.tar.gz (download from https://iupred3.elte.hu/):')
        _up2 = _files.upload()
        if not _up2:
            print('=' * 60)
            print('  ⚠  CMAscan: Invalid FASTA file')
            print('=' * 60)
            print()
            print('No IUPred3 archive uploaded.')
            print()
            print('Please correct your file and run the pipeline again.')
            print('Reload via Step 1 (Choose Files) after fixing.')
            print('=' * 60)
            raise SystemExit(1)
        _fname2 = list(_up2.keys())[0]
        _raw = _up2[_fname2]
        with open(IUPRED3_ARCHIVE_PATH, 'wb') as _f2:
            _f2.write(_raw)
        # Validate: check it is a tar.gz
        import tarfile as _tf
        if not _tf.is_tarfile(IUPRED3_ARCHIVE_PATH):
            print('=' * 60)
            print('  ⚠  Uploaded file does not appear to be iupred3.tar.gz')
            print('=' * 60)
            print('Please download from https://iupred3.elte.hu/download_new')
            raise SystemExit(1)
        print(f'IUPred3 archive saved: {IUPRED3_ARCHIVE_PATH}')
    else:
        print(f'IUPred3 archive found: {IUPRED3_ARCHIVE_PATH}')
else:
    IUPRED3_ARCHIVE_PATH = '/content/iupred3.tar.gz'

_output_name = (OUTPUT_FILE_NAME or '').strip()
if not _output_name:
    _output_name = 'motif_hits_scored.csv'
if not _output_name.lower().endswith('.csv'):
    _output_name = f'{_output_name}.csv'
OUTPUT_FILE_PATH = f'/content/{_output_name}'


This step defines the input source and user-facing options used by the full pipeline.


In [ ]:
#@title Step 2: Imports and Global Settings
import csv
import json
import pickle
import subprocess
import tarfile
import tempfile
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterator, List, Sequence
from urllib.parse import quote

from google.colab import files
import pandas as pd

ALLOWED_EXTENSIONS = {'.fasta', '.faa', '.txt'}
WINDOW_SIZE = 5
VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

ALLOWED_AMIDE = {'Q', 'N'}
ALLOWED_REMAINDER = set('EDRKFLIVSTY')
NEGATIVE_RESIDUES = {'E', 'D'}
POSITIVE_RESIDUES = {'R', 'K'}
HYDROPHOBIC_RESIDUES = {'F', 'L', 'I', 'V'}
PHOSPHO_RESIDUES = {'S', 'T', 'Y'}

PSSM_URL_CANDIDATES = {
    'cPSSM': ['https://raw.githubusercontent.com/Marcin-Lubocki/CMAscan/main/PSSM/cPSSM.pkl'],
    'ePSSM': ['https://raw.githubusercontent.com/Marcin-Lubocki/CMAscan/main/PSSM/ePSSM.pkl'],
}

# MusiteDeep integration: planned for v1.1

IUPRED3_MODE = 'long'
IUPRED3_MIN_SEQUENCE_LENGTH = 30
IUPRED3_TIMEOUT_SEC = 180

# Default pseudo-PTM options (overwritten by form cell before run).
USE_PSEUDOACETYLATION = True
USE_PSEUDOPHOSPHORYLATION = True
PTM_OFF_VALUE = 'Off'

C_PSSM_MATRIX: Any | None = None
E_PSSM_MATRIX: Any | None = None

Core libraries and global constants are loaded to keep the run reproducible.


In [ ]:
#@title Step 3: Install BioPython
%pip install -q biopython tqdm
print('Dependencies installed: biopython, tqdm')

BioPython is installed in the current Colab runtime (run once per fresh session).


In [ ]:
#@title Step 4: Data Models and Input Validation
import re
import sys


def _friendly_fasta_error(message: str, details: str = '') -> None:
    """Print a clearly formatted error message and exit gracefully."""
    banner = '=' * 60
    print(banner)
    print('  ⚠  CMAscan: Invalid FASTA file')
    print(banner)
    print()
    print(message)
    if details:
        print()
        print(details)
    print()
    print('Please correct your file and run the pipeline again.')
    print('Reload via Step 1 (Choose Files) after fixing.')
    print(banner)
    raise SystemExit(1)


@dataclass(frozen=True)
class FastaRecord:
    header: str
    sequence: str

    @property
    def protein_name(self) -> str:
        return self.header.split()[0]


@dataclass(frozen=True)
class MotifCandidate:
    original_mer: str
    oriented_mer: str
    localization: int
    window_start_idx: int
    reoriented: bool


def include_ptm_simulation_columns() -> bool:
    return USE_PSEUDOACETYLATION or USE_PSEUDOPHOSPHORYLATION


def validate_ptm_options() -> None:
    if PSEUDOPHOSPHO_RESIDUE not in {'E', 'D'}:
        raise ValueError("PSEUDOPHOSPHO_RESIDUE must be 'E' or 'D'.")


# MusiteDeep integration: planned for v1.1

def print_ptm_simulation_summary() -> None:
    label = 'Pseudo-PTM simulation for PSSM/IUPred3' if USE_IUPRED3 else 'Pseudo-PTM simulation for PSSM'
    if not include_ptm_simulation_columns():
        print(f'{label}: OFF')
        return
    acetyl_text = 'ON (K->Q)' if USE_PSEUDOACETYLATION else f'OFF ({PTM_OFF_VALUE})'
    phospho_text = (
        f'ON (S/T/Y->{PSEUDOPHOSPHO_RESIDUE})'
        if USE_PSEUDOPHOSPHORYLATION
        else f'OFF ({PTM_OFF_VALUE})'
    )
    print(f'{label}: acetyl={acetyl_text}; phospho={phospho_text}')


def validate_input_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Input file not found: {path}')
    if path.suffix.lower() not in ALLOWED_EXTENSIONS:
        allowed = ', '.join(sorted(ALLOWED_EXTENSIONS))
        raise ValueError(f'Invalid input format: {path.suffix}. Allowed: {allowed}')


def parse_multifasta(path: Path) -> List[FastaRecord]:
    records: List[FastaRecord] = []
    current_header: str | None = None
    current_seq_chunks: List[str] = []

    with path.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_header is not None:
                    raw_seq = ''.join(current_seq_chunks)
                    clean_seq = re.sub(r'\s+', '', raw_seq).upper()
                    records.append(FastaRecord(current_header, clean_seq))
                current_header = line[1:].strip()
                current_seq_chunks = []
            else:
                current_seq_chunks.append(line)

    if current_header is not None:
        raw_seq = ''.join(current_seq_chunks)
        clean_seq = re.sub(r'\s+', '', raw_seq).upper()
        records.append(FastaRecord(current_header, clean_seq))

    if not records:
        _friendly_fasta_error(
            'No FASTA records were found in the uploaded file.',
            'Each protein must start with a header line beginning with ">".\n'
            'If you pasted sequences from Word or PDF, the file may have lost\n'
            'its formatting. Try saving as plain text (.txt) or FASTA (.fasta).'
        )

    return records

def sanity_check_records(records: Sequence[FastaRecord], allow_x: bool = True) -> List[FastaRecord]:
    invalid_entries: List[str] = []
    valid_records: List[FastaRecord] = []

    for rec in records:
        if not rec.sequence:
            invalid_entries.append(f'{rec.protein_name}: empty sequence')
            continue

        bad_residues = sorted({aa for aa in rec.sequence if aa not in VALID_AA and not (allow_x and aa == 'X')})
        if bad_residues:
            invalid_entries.append(f"{rec.protein_name}: invalid residue(s) {','.join(bad_residues)}")
            continue

        valid_records.append(rec)

    if invalid_entries and not valid_records:
        _friendly_fasta_error(
            'All sequences in the uploaded file contain non-standard residues and could not be analyzed.',
            'CMAscan accepts only the 20 standard amino acids\n'
            '(ACDEFGHIKLMNPQRSTVWY) plus X for unknown residues.\n'
            'Common issues:\n'
            '  - Numbers in sequence (e.g. "1MAGEKLA") — remove residue numbering\n'
            '  - Hyphens or dots — these indicate alignment files, not raw FASTA\n'
            '  - DNA/RNA sequences — CMAscan analyzes proteins only\n'
            '  - Stop codon symbols (*) — remove these before submission'
        )

    if invalid_entries:
        print(f'⚠  Warning: {len(invalid_entries)} sequence(s) skipped due to non-standard residues:')
        for entry in invalid_entries[:10]:
            print(f'   - {entry}')
        if len(invalid_entries) > 10:
            print(f'   ... and {len(invalid_entries) - 10} more')
        print(f'Continuing analysis on {len(valid_records)} valid sequence(s).')
        print()

    if not valid_records:
        _friendly_fasta_error(
            'All sequences in the uploaded file contain non-standard residues and could not be analyzed.',
            'CMAscan accepts only the 20 standard amino acids\n'
            '(ACDEFGHIKLMNPQRSTVWY) plus X for unknown residues.\n'
            'Common issues:\n'
            '  - Numbers in sequence (e.g. "1MAGEKLA") — remove residue numbering\n'
            '  - Hyphens or dots — these indicate alignment files, not raw FASTA\n'
            '  - DNA/RNA sequences — CMAscan analyzes proteins only\n'
            '  - Stop codon symbols (*) — remove these before submission'
        )

    return valid_records

FASTA parsing and sanity checks validate protein sequences before scoring.


In [ ]:
#@title Step 5: Sliding Window and Orientation Rules
def iter_windows(sequence: str, k: int = WINDOW_SIZE) -> Iterator[tuple[int, str]]:
    if len(sequence) < k:
        return
    for start_idx in range(0, len(sequence) - k + 1):
        yield start_idx, sequence[start_idx:start_idx + k]


def reorient(window: str) -> str:
    return window[::-1]


def classify_window(window: str, start_idx: int) -> List[MotifCandidate]:
    candidates: List[MotifCandidate] = []
    amide_positions = [i for i, aa in enumerate(window) if aa in ALLOWED_AMIDE]
    amide_count = len(amide_positions)

    if amide_count == 1:
        amide_pos = amide_positions[0]
        if amide_pos == 0:
            candidates.append(MotifCandidate(window, window, start_idx + 1, start_idx, False))
        elif amide_pos == WINDOW_SIZE - 1:
            candidates.append(MotifCandidate(window, reorient(window), start_idx + 1, start_idx, True))
        return candidates

    if amide_count == 0:
        first_is_k = window[0] == 'K'
        last_is_k = window[-1] == 'K'

        if first_is_k and last_is_k:
            candidates.append(MotifCandidate(window, window, start_idx + 1, start_idx, False))
            candidates.append(MotifCandidate(window, reorient(window), start_idx + 1, start_idx, True))
            return candidates

        if first_is_k:
            candidates.append(MotifCandidate(window, window, start_idx + 1, start_idx, False))
            return candidates

        if last_is_k:
            candidates.append(MotifCandidate(window, reorient(window), start_idx + 1, start_idx, True))
            return candidates

    return candidates

A 5-residue sliding window with orientation rules generates motif candidates.


In [ ]:
#@title Step 6: Strict Motif Type Classification
def summarize_remainder(remainder: str) -> Dict[str, int | bool]:
    return {
        'all_allowed': all(aa in ALLOWED_REMAINDER for aa in remainder),
        'negative_count': sum(1 for aa in remainder if aa in NEGATIVE_RESIDUES),
        'positive_count': sum(1 for aa in remainder if aa in POSITIVE_RESIDUES),
        'hydrophobic_count': sum(1 for aa in remainder if aa in HYDROPHOBIC_RESIDUES),
        'phospho_count': sum(1 for aa in remainder if aa in PHOSPHO_RESIDUES),
    }


def classify_motif_type(oriented_mer: str) -> str | None:
    amide_positions = [i for i, aa in enumerate(oriented_mer) if aa in ALLOWED_AMIDE]
    amide_count = len(amide_positions)

    if amide_count == 0:
        if oriented_mer[0] != 'K':
            return None
        stats = summarize_remainder(oriented_mer[1:])
        if not stats['all_allowed']:
            return None
        if stats['negative_count'] != 1:
            return None
        if stats['positive_count'] < 1:
            return None
        if stats['hydrophobic_count'] < 1:
            return None
        if stats['phospho_count'] != 0:
            return None
        return 'acetyl'

    if amide_count == 1 and amide_positions[0] == 0:
        stats = summarize_remainder(oriented_mer[1:])
        if not stats['all_allowed']:
            return None

        is_canonical = (
            stats['negative_count'] == 1
            and stats['positive_count'] >= 1
            and stats['hydrophobic_count'] >= 1
            and stats['phospho_count'] == 0
        )
        if is_canonical:
            return 'canonical'

        is_phospho = (
            stats['phospho_count'] == 1
            and stats['positive_count'] >= 1
            and stats['hydrophobic_count'] >= 1
            and stats['negative_count'] == 0
        )
        if is_phospho:
            return 'phospho'

    return None

Motif candidates are classified as canonical, phospho, or acetyl using strict composition rules.


In [ ]:
#@title Step 7: Load cPSSM and ePSSM
def load_pickle_from_url(url: str) -> Any:
    with urllib.request.urlopen(url, timeout=30) as response:
        return pickle.loads(response.read())


def load_pssm_matrix(url_candidates: Sequence[str], matrix_name: str) -> tuple[Any, str]:
    errors: List[str] = []
    for url in url_candidates:
        try:
            return load_pickle_from_url(url), url
        except Exception as exc:  # noqa: BLE001
            errors.append(f'{url} -> {exc}')
    raise RuntimeError('Failed to download ' + matrix_name + '. Tried: ' + '; '.join(errors))


def ensure_pssm_matrices_loaded() -> None:
    global C_PSSM_MATRIX, E_PSSM_MATRIX

    try:
        from Bio import motifs as _bio_motifs  # noqa: F401
    except Exception as exc:  # noqa: BLE001
        raise RuntimeError('BioPython is required. Run the installation cell first.') from exc

    if C_PSSM_MATRIX is None:
        C_PSSM_MATRIX, c_url = load_pssm_matrix(PSSM_URL_CANDIDATES['cPSSM'], 'cPSSM')
        print(f'Loaded cPSSM from {c_url}')

    if E_PSSM_MATRIX is None:
        E_PSSM_MATRIX, e_url = load_pssm_matrix(PSSM_URL_CANDIDATES['ePSSM'], 'ePSSM')
        print(f'Loaded ePSSM from {e_url}')

---

IUPred3 disorder scores are computed using the IUPred3 tool
(https://iupred3.elte.hu/), an external program with its own academic
license. CMAscan does NOT bundle or redistribute IUPred3.

To use IUPred3 with CMAscan:
1. Visit https://iupred3.elte.hu/download_new and accept the academic
   license to download `iupred3.tar.gz` (free for academic users).
2. Upload the file to `/content/iupred3.tar.gz` in Colab.
3. Set `USE_IUPRED3 = True` in Step 1.

When `USE_IUPRED3 = False` (default), this step is skipped and the
pipeline runs without disorder scoring.

**If you use CMAscan output in a publication, please cite the IUPred3
authors:**

- Erdős G, Pajkos M, Dosztányi Z. IUPred3: prediction of protein disorder
  enhanced with unambiguous experimental annotation and visualization of
  evolutionary conservation. *Nucleic Acids Research* 2021;49(W1):W297–W303.

- Mészáros B, Erdős G, Dosztányi Z. IUPred2A: context-dependent prediction
  of protein disorder as a function of redox state and protein binding.
  *Nucleic Acids Research* 2018;46(W1):W329–W337.

- Erdős G, Dosztányi Z. Analyzing Protein Disorder with IUPred2A.
  *Current Protocols in Bioinformatics* 2020;70(1):e99.

---

In [ ]:
#@title Step 8: IUPred3 Disorder Scoring
def default_iupred_install_root() -> Path:
    return Path('/content/iupred3')


def safe_extract_tar_gz(tar_path: Path, target_dir: Path) -> None:
    target_dir_resolved = target_dir.resolve()
    with tarfile.open(tar_path, 'r:gz') as tar:
        for member in tar.getmembers():
            member_path = (target_dir / member.name).resolve()
            if not str(member_path).startswith(str(target_dir_resolved)):
                raise RuntimeError(f'Unsafe tar member path: {member.name}')
        tar.extractall(target_dir)


def ensure_iupred3_script() -> Path:
    archive_path = Path(IUPRED3_ARCHIVE_PATH)
    if not archive_path.is_file():
        raise RuntimeError(f'IUPred3 archive not found at {archive_path}. Upload iupred3.tar.gz and set USE_IUPRED3=True.')

    install_root = default_iupred_install_root()
    install_root.mkdir(parents=True, exist_ok=True)
    script = install_root / 'iupred3' / 'iupred3.py'
    if script.exists():
        return script

    print(f'IUPred3: extracting from {archive_path}')
    safe_extract_tar_gz(archive_path, install_root)

    if not script.exists():
        raise RuntimeError(f'IUPred3 script not found after extraction: {script}')
    return script


def parse_iupred3_output(raw_output: str) -> Dict[int, float]:
    scores: Dict[int, float] = {}
    for line in raw_output.splitlines():
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('POS'):
            continue
        parts = line.split()
        if len(parts) < 3:
            continue
        try:
            pos = int(parts[0])
            val = float(parts[2])
        except ValueError:
            continue
        scores[pos] = val
    return scores


def run_iupred3_long(sequence: str, iupred_script: Path) -> Dict[int, float]:
    if len(sequence) < IUPRED3_MIN_SEQUENCE_LENGTH:
        return {}

    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.fasta') as tmp:
        tmp.write(f'>query\n{sequence}\n')
        tmp_path = Path(tmp.name)

    command = ['python3', str(iupred_script), str(tmp_path), IUPRED3_MODE]
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True,
            timeout=IUPRED3_TIMEOUT_SEC,
            cwd=str(iupred_script.parent),
        )
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or '').strip()
        stdout = (exc.stdout or '').strip()
        raise RuntimeError(f'IUPred3 failed: {stderr or stdout or exc}') from exc
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f'IUPred3 timed out after {IUPRED3_TIMEOUT_SEC}s') from exc
    finally:
        tmp_path.unlink(missing_ok=True)

    return parse_iupred3_output(result.stdout)


def build_iupred3_cache(records: Sequence[FastaRecord], iupred_script: Path) -> Dict[str, Dict[int, float]]:
    cache: Dict[str, Dict[int, float]] = {}
    print(f'IUPred3 mode: {IUPRED3_MODE}')
    print(f'IUPred3 script: {iupred_script}')

    for rec in records:
        if len(rec.sequence) < IUPRED3_MIN_SEQUENCE_LENGTH:
            cache[rec.protein_name] = {}
            continue

        try:
            cache[rec.protein_name] = run_iupred3_long(rec.sequence, iupred_script)
        except Exception as exc:  # noqa: BLE001
            print(f'Warning: IUPred3 failed for {rec.protein_name}: {exc}')
            cache[rec.protein_name] = {}

    return cache


def compute_iupred3_window_mean(iupred_scores: Dict[int, float], window_start_idx: int, window_size: int = WINDOW_SIZE) -> float | None:
    positions = range(window_start_idx + 1, window_start_idx + window_size + 1)
    values = [iupred_scores[pos] for pos in positions if pos in iupred_scores]
    if len(values) != window_size:
        return None
    return float(sum(values) / len(values))


if not USE_IUPRED3:
    print('IUPred3: disabled.')

IUPred3 helper functions are defined above and run only when `USE_IUPRED3 = True`.

In [ ]:
#@title Step 9: PSSM + PTM Mimetic Helpers
def oriented_index_to_sequence_position(candidate: MotifCandidate, oriented_index: int) -> int:
    if candidate.reoriented:
        original_offset = (WINDOW_SIZE - 1) - oriented_index
    else:
        original_offset = oriented_index
    return candidate.window_start_idx + original_offset + 1

def get_ptm_target_position(sequence: str, candidate: MotifCandidate, motif_type: str) -> tuple[int, str] | None:
    if motif_type == 'phospho':
        sty_indices = [i for i, aa in enumerate(candidate.oriented_mer) if aa in PHOSPHO_RESIDUES]
        if len(sty_indices) != 1:
            return None
        pos_1b = oriented_index_to_sequence_position(candidate, sty_indices[0])
        if not (1 <= pos_1b <= len(sequence)):
            return None
        residue = sequence[pos_1b - 1]
        if residue not in PHOSPHO_RESIDUES:
            return None
        return pos_1b, residue

    if motif_type == 'acetyl':
        pos_1b = oriented_index_to_sequence_position(candidate, 0)
        if not (1 <= pos_1b <= len(sequence)):
            return None
        residue = sequence[pos_1b - 1]
        if residue != 'K':
            return None
        return pos_1b, residue

    return None

def get_pssm_position_score(pssm_matrix: Any, aa: str, pos: int) -> float:
    try:
        return float(pssm_matrix[aa][pos])
    except Exception:  # noqa: BLE001
        pass
    if hasattr(pssm_matrix, 'pssm'):
        return get_pssm_position_score(pssm_matrix.pssm, aa, pos)
    if isinstance(pssm_matrix, dict) and aa in pssm_matrix:
        return float(pssm_matrix[aa][pos])
    raise KeyError(f"Amino acid {aa} missing in PSSM")


def compute_pssm_score(mer: str, pssm_matrix: Any) -> float:
    motif = mer.replace('U', 'C')
    score = 0.0
    for i, aa in enumerate(motif):
        try:
            score += get_pssm_position_score(pssm_matrix, aa, i)
        except KeyError:
            score += 0.0
    return float(score)


def replace_residue_at_position(sequence: str, position_1b: int, new_residue: str) -> str:
    idx = position_1b - 1
    return sequence[:idx] + new_residue + sequence[idx + 1:]


def get_phospho_oriented_index(oriented_mer: str) -> int | None:
    indices = [i for i, aa in enumerate(oriented_mer) if aa in PHOSPHO_RESIDUES]
    if len(indices) != 1:
        return None
    return indices[0]


def build_ptm_mimetic_mers(candidate: MotifCandidate, motif_type: str) -> tuple[str, str] | None:
    if motif_type == 'acetyl':
        replacement = 'Q'
        target_index = 0
    elif motif_type == 'phospho':
        target_index = get_phospho_oriented_index(candidate.oriented_mer)
        if target_index is None:
            return None
        replacement = PSEUDOPHOSPHO_RESIDUE
    else:
        return None

    oriented_chars = list(candidate.oriented_mer)
    oriented_chars[target_index] = replacement
    oriented_mimetic = ''.join(oriented_chars)
    original_mimetic = oriented_mimetic[::-1] if candidate.reoriented else oriented_mimetic
    return original_mimetic, oriented_mimetic


def compute_iupred3_ptm_window_mean(
    sequence: str,
    window_start_idx: int,
    target_pos_1b: int,
    replacement_residue: str,
    iupred_script: Path,
    cache: Dict[tuple[str, int, str], Dict[int, float]],
) -> float | None:
    if len(sequence) < IUPRED3_MIN_SEQUENCE_LENGTH:
        return None

    cache_key = (sequence, target_pos_1b, replacement_residue)
    if cache_key not in cache:
        mutated_sequence = replace_residue_at_position(sequence, target_pos_1b, replacement_residue)
        try:
            cache[cache_key] = run_iupred3_long(mutated_sequence, iupred_script)
        except Exception as exc:  # noqa: BLE001
            print(f'Warning: IUPred3 PTM rerun failed (pos={target_pos_1b}, replace={replacement_residue}): {exc}')
            cache[cache_key] = {}

    return compute_iupred3_window_mean(cache[cache_key], window_start_idx, WINDOW_SIZE)

Helper functions define pseudo-PTM mimetics and rescoring logic.


In [ ]:
#@title Step 10: Build Pipeline and Export CSV
from tqdm.notebook import tqdm
def find_all_motif_hits(
    records: Sequence[FastaRecord],
    use_iupred3: bool,
    iupred3_cache: Dict[str, Dict[int, float]] | None = None,
    iupred_script: Path | None = None,
) -> List[Dict[str, str | int | float]]:
    rows: List[Dict[str, str | int | float]] = []
    use_ptm_columns = include_ptm_simulation_columns()
    ptm_iupred_cache: Dict[tuple[str, int, str], Dict[int, float]] = {}

    for rec in tqdm(records, desc='Scanning proteins', unit='protein'):
        record_hits = 0
        rec_iupred_scores = iupred3_cache.get(rec.protein_name, {}) if (use_iupred3 and iupred3_cache is not None) else {}

        for start_idx, window in iter_windows(rec.sequence, WINDOW_SIZE):
            candidates = classify_window(window, start_idx)
            for cand in candidates:
                motif_type = classify_motif_type(cand.oriented_mer)
                if motif_type is None:
                    continue

                target = get_ptm_target_position(rec.sequence, cand, motif_type)
                # MusiteDeep integration: planned for v1.1

                iupred_value = compute_iupred3_window_mean(rec_iupred_scores, cand.window_start_idx, WINDOW_SIZE) if use_iupred3 else None

                mer_ptm_mimetic: str | float = 'NA'
                c_pssm_ptm: str | float = 'NA'
                e_pssm_ptm: str | float = 'NA'
                iupred3_ptm: str | float = 'NA'

                if use_ptm_columns:
                    if motif_type == 'canonical':
                        mer_ptm_mimetic = 'NA'
                        c_pssm_ptm = 'NA'
                        e_pssm_ptm = 'NA'
                        iupred3_ptm = 'NA'
                    elif motif_type == 'acetyl' and not USE_PSEUDOACETYLATION:
                        mer_ptm_mimetic = PTM_OFF_VALUE
                        c_pssm_ptm = PTM_OFF_VALUE
                        e_pssm_ptm = PTM_OFF_VALUE
                        iupred3_ptm = PTM_OFF_VALUE if use_iupred3 else 'NA'
                    elif motif_type == 'phospho' and not USE_PSEUDOPHOSPHORYLATION:
                        mer_ptm_mimetic = PTM_OFF_VALUE
                        c_pssm_ptm = PTM_OFF_VALUE
                        e_pssm_ptm = PTM_OFF_VALUE
                        iupred3_ptm = PTM_OFF_VALUE if use_iupred3 else 'NA'
                    else:
                        mimetic = build_ptm_mimetic_mers(cand, motif_type)
                        if mimetic is not None and target is not None:
                            mer_ptm_mimetic, oriented_mimetic_mer = mimetic
                            c_pssm_ptm = round(compute_pssm_score(oriented_mimetic_mer, C_PSSM_MATRIX), 2)
                            e_pssm_ptm = round(compute_pssm_score(oriented_mimetic_mer, E_PSSM_MATRIX), 2)

                            if use_iupred3 and iupred_script is not None:
                                ptm_pos_1b, _ = target
                                replacement_residue = 'Q' if motif_type == 'acetyl' else PSEUDOPHOSPHO_RESIDUE
                                ptm_iupred_value = compute_iupred3_ptm_window_mean(
                                    rec.sequence,
                                    cand.window_start_idx,
                                    ptm_pos_1b,
                                    replacement_residue,
                                    iupred_script,
                                    ptm_iupred_cache,
                                )
                                iupred3_ptm = round(ptm_iupred_value, 2) if ptm_iupred_value is not None else 'NA'
                            else:
                                iupred3_ptm = 'NA'

                row = {
                    'protein_name': rec.protein_name,
                    'mer': cand.original_mer,
                    'type': motif_type,
                    'localization': int(cand.localization),
                    'cPSSM': round(compute_pssm_score(cand.oriented_mer, C_PSSM_MATRIX), 2),
                    'ePSSM': round(compute_pssm_score(cand.oriented_mer, E_PSSM_MATRIX), 2),
                    'IUPred3': round(iupred_value, 2) if iupred_value is not None else 'NA',
                    # MusiteDeep integration: planned for v1.1
                }
                if use_ptm_columns:
                    row['mer_PTM_mimetic'] = mer_ptm_mimetic
                    row['cPSSM_PTM'] = c_pssm_ptm
                    row['ePSSM_PTM'] = e_pssm_ptm
                    row['IUPred3_PTM'] = iupred3_ptm

                rows.append(row)
                record_hits += 1

        if record_hits == 0:
            empty_row: Dict[str, str | int | float] = {
                'protein_name': rec.protein_name,
                'mer': 'NA',
                'type': 'No motifs found',
                'localization': 'NA',
                'cPSSM': 'NA',
                'ePSSM': 'NA',
                'IUPred3': 'NA',
                # MusiteDeep integration: planned for v1.1
            }
            if use_ptm_columns:
                empty_row['mer_PTM_mimetic'] = 'NA'
                empty_row['cPSSM_PTM'] = 'NA'
                empty_row['ePSSM_PTM'] = 'NA'
                empty_row['IUPred3_PTM'] = 'NA'
            rows.append(empty_row)

    return rows


def write_csv(rows: Sequence[Dict[str, str | int | float]], output_path: Path) -> None:
    columns = ['protein_name', 'mer', 'type', 'localization', 'cPSSM', 'ePSSM', 'IUPred3']
    if include_ptm_simulation_columns():
        columns.extend(['mer_PTM_mimetic', 'cPSSM_PTM', 'ePSSM_PTM', 'IUPred3_PTM'])
    # MusiteDeep integration: planned for v1.1

    with output_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def run_pipeline(input_path: str, output_csv: str = 'motif_hits_scored.csv') -> Path:
    validate_ptm_options()
    print_ptm_simulation_summary()

    in_path = Path(input_path)
    out_path = Path(output_csv)

    validate_input_file(in_path)
    records = parse_multifasta(in_path)
    records = sanity_check_records(records, allow_x=True)

    ensure_pssm_matrices_loaded()

    if USE_IUPRED3:
        iupred_script = ensure_iupred3_script()
        iupred3_cache = build_iupred3_cache(records, iupred_script)
    else:
        print('IUPred3: disabled.')
        iupred_script = None
        iupred3_cache = None
    # MusiteDeep integration: planned for v1.1

    rows = find_all_motif_hits(records, USE_IUPRED3, iupred3_cache, iupred_script)
    write_csv(rows, out_path)

    print()
    print('--- CMAscan run complete ---')
    print(f'  Proteins processed : {len(records)}')
    motif_rows = [r for r in rows if r.get("type") != "No motifs found"]
    print(f'  Motif hits found   : {len(motif_rows)}')
    print(f'  Rows in CSV        : {len(rows)}')
    print(f'  Output file        : {out_path.resolve()}')
    print(f'  IUPred3 scoring    : {"enabled" if USE_IUPRED3 else "disabled"}')
    print()
    print('If results look incomplete or an error occurred, click')
    print('Runtime -> Disconnect and delete runtime, then Runtime -> Run all.')

    return out_path

The end-to-end pipeline function is assembled and ready to execute.


In [ ]:
#@title Step 11: Run Analysis
result_path = run_pipeline(input_path=INPUT_FILE_PATH, output_csv=OUTPUT_FILE_PATH)

The full analysis is executed on the selected FASTA input and exported as CSV.


In [ ]:
#@title Step 12: Preview and Download CSV
df = pd.read_csv(result_path)
display(df.head(30))
print(f'Total rows: {len(df)}')
print('Type distribution:')
print(df['type'].value_counts(dropna=False))
print()
print('Output column legend:')
legend = {
    'protein_name': 'Identifier from FASTA header',
    'mer': '5-residue motif candidate in original sequence orientation',
    'type': 'canonical | phospho | acetyl | "No motifs found"',
    'localization': '1-based start position of the motif in the protein sequence',
    'cPSSM': 'Score from canonical PSSM (canonical CMA motifs)',
    'ePSSM': 'Score from extended PSSM (canonical + atypical CMA motifs)',
    'IUPred3': 'Mean disorder score over the 5 motif residues (0 = ordered, 1 = disordered)',
    'mer_PTM_mimetic': 'Motif sequence after simulated PTM (K -> Q for acetyl; S/T/Y -> E or D for phospho)',
    'cPSSM_PTM': 'cPSSM score recomputed for the PTM mimetic motif',
    'ePSSM_PTM': 'ePSSM score recomputed for the PTM mimetic motif',
    'IUPred3_PTM': 'Disorder score recomputed on sequence with PTM mimetic substitution',
}
for col, desc in legend.items():
    if col in df.columns:
        print(f'  {col}: {desc}')
files.download(str(result_path))

# # Visualizations
This section provides visual summaries of the CMAscan analysis results.

In [ ]:
#@title Visualization Setup (Constants and Mappings)
# ============================================================
# Visualization guard — skip all plots if no motifs were found
# ============================================================
try:
    _viz_df = pd.read_csv(OUTPUT_FILE_PATH)
except Exception as exc:
    print(f'Visualization skipped: cannot read output CSV ({exc}).')
    raise SystemExit(0)

_real_hits = _viz_df[_viz_df['type'] != 'No motifs found']
if _real_hits.empty:
    print('=' * 60)
    print('  Visualization section')
    print('=' * 60)
    print()
    print('No motif hits were found in any of the input sequences.')
    print('Skipping visualization (no data to plot).')
    print()
    print('You can still review the output CSV — it lists each input')
    print('protein with type = "No motifs found".')
    raise SystemExit(0)
else:
    print(f'Visualization: rendering plots for {len(_real_hits)} motif hits '
          f'across {_real_hits["protein_name"].nunique()} proteins.')

# Continue with original cell contents below

# Define consistent colorblind palette
MOTIF_COLORS = {
    'canonical': '#009E73',
    'phospho': '#0072B2',
    'acetyl': '#F0E442'
}

# Ensure sequence lengths are mapped for density calculations
try:
    length_map = {rec.protein_name: len(rec.sequence) for rec in records}
except NameError:
    # Fallback if records is not in memory
    from pathlib import Path
    records_temp = parse_multifasta(Path(INPUT_FILE_PATH))
    length_map = {rec.protein_name: len(rec.sequence) for rec in records_temp}

print("Visualization constants and length maps initialized.")

In [ ]:
#@title 1. Frequency of Identified CMA Motif Types
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
type_counts = df[df['type'] != 'No motifs found']['type'].value_counts().reset_index()
type_counts.columns = ['Motif Type', 'Frequency']

plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Motif Type', y='Frequency', data=type_counts, palette=MOTIF_COLORS, hue='Motif Type', legend=False)
plt.title('Frequency of Identified CMA Motif Types', fontsize=14)
for p in ax.patches: ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

In [ ]:
#@title 2. Density Analysis (Hits/100aa)
hits_per_prot = df[df['type'] != 'No motifs found'].groupby(['protein_name', 'type']).size().reset_index(name='count')
hits_per_prot['length'] = hits_per_prot['protein_name'].map(length_map)
hits_per_prot['density_100'] = (hits_per_prot['count'] / hits_per_prot['length']) * 100

plt.figure(figsize=(12, 6))
sns.boxplot(data=hits_per_prot, x='type', y='density_100', palette=MOTIF_COLORS, hue='type', showfliers=False, boxprops={'alpha': 0.3})
sns.stripplot(data=hits_per_prot, x='type', y='density_100', palette=MOTIF_COLORS, hue='type', jitter=True, size=8)
plt.title('Motif Density Analysis (Hits per 100 residues)')
plt.show()

In [ ]:
#@title 3. Motif Density Heatmap
HEATMAP_PROTEIN_LIMIT = 100

heatmap_data = hits_per_prot.pivot(index='protein_name', columns='type', values='density_100').fillna(0)

total_proteins = len(heatmap_data)
if total_proteins > HEATMAP_PROTEIN_LIMIT:
    print(f'Note: input contains {total_proteins} proteins. The heatmap '
          f'displays the first {HEATMAP_PROTEIN_LIMIT} proteins (by FASTA '
          f'input order). The full output CSV contains all proteins.')
    heatmap_data = heatmap_data.head(HEATMAP_PROTEIN_LIMIT)

# Dynamic height: 0.3 inch per protein, minimum 8 inches
fig_height = max(8, len(heatmap_data) * 0.3)
plt.figure(figsize=(12, fig_height))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Heatmap: Motif Density by Protein and Type')
plt.tight_layout()
plt.show()

In [ ]:
#@title Swarm Distribution Helper
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plot_swarm_distribution(data, score_col, ptm_score_col, title, xlabel):
    """Plot a strip distribution of score values grouped by motif type."""
    plot_df = data[data['type'] != 'No motifs found'].copy()

    # Helper to select the correct score based on type
    def get_final_score(row):
        if row['type'] == 'canonical' or ptm_score_col is None:
            return pd.to_numeric(row[score_col], errors='coerce')
        else:
            return pd.to_numeric(row[ptm_score_col], errors='coerce')

    plot_df['final_score'] = plot_df.apply(get_final_score, axis=1)

    plt.figure(figsize=(12, 10))
    sns.stripplot(
        data=plot_df,
        x='final_score',
        y='protein_name',
        hue='type',
        palette=MOTIF_COLORS,  # Applied hardcoded colors
        size=8,
        alpha=0.7,
        linewidth=1,
        edgecolor='auto',
        jitter=0.25,
    )

    for i in range(len(plot_df['protein_name'].unique())):
        plt.axhline(i, color='gray', linestyle='--', alpha=0.2, zorder=0)

    plt.title(title, fontsize=14)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel('Protein Name', fontsize=12)
    plt.legend(title='Motif Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='x', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 4a. cPSSM Distribution
plot_swarm_distribution(df, 'cPSSM', 'cPSSM_PTM', 'Distribution of cPSSM Scores', 'cPSSM Score')

In [ ]:
#@title 4b. ePSSM Swarm Distribution
plot_swarm_distribution(df, 'ePSSM', 'ePSSM_PTM', 'Distribution of ePSSM Scores', 'ePSSM Score')

In [ ]:
#@title 4c. IUPred3 Swarm Distribution
if not USE_IUPRED3:
    print('=' * 60)
    print('  Skipped: IUPred3 Swarm Distribution')
    print('=' * 60)
    print()
    print('This visualization requires IUPred3 disorder scoring.')
    print('To enable, set USE_IUPRED3 = True in Configuration and re-run')
    print('the notebook (after uploading iupred3.tar.gz to /content/).')
    print()
else:
    plot_swarm_distribution(df, 'IUPred3', 'IUPred3_PTM', 'Distribution of IUPred3 Scores', 'IUPred3 Score')

In [ ]:
#@title 5. Interactive 3D Map of Identified Motif Hits
if not USE_IUPRED3:
    print('=' * 60)
    print('  Skipped: Interactive 3D Map of Identified Motif Hits')
    print('=' * 60)
    print()
    print('This visualization requires IUPred3 disorder scoring.')
    print('To enable, set USE_IUPRED3 = True in Configuration and re-run')
    print('the notebook (after uploading iupred3.tar.gz to /content/).')
    print()
else:
    import plotly.express as px

    # Prepare data for 3D plot
    plot_3d_df = df[df['type'] != 'No motifs found'].copy()
    plot_3d_df['cPSSM_final'] = plot_3d_df.apply(lambda r: r['cPSSM_PTM'] if r['type'] != 'canonical' else r['cPSSM'], axis=1)
    plot_3d_df['ePSSM_final'] = plot_3d_df.apply(lambda r: r['ePSSM_PTM'] if r['type'] != 'canonical' else r['ePSSM'], axis=1)
    plot_3d_df['IUPred3_final'] = plot_3d_df.apply(lambda r: r['IUPred3_PTM'] if r['type'] != 'canonical' else r['IUPred3'], axis=1)

    fig = px.scatter_3d(
        plot_3d_df,
        x='cPSSM_final',
        y='ePSSM_final',
        z='IUPred3_final',
        color='type',
        color_discrete_map=MOTIF_COLORS,  # Applied hardcoded colors
        hover_data=['protein_name', 'mer', 'localization'],
        title='3D Map of CMA Motif Scores',
        labels={'cPSSM_final': 'cPSSM (PTM-adj)', 'ePSSM_final': 'ePSSM (PTM-adj)', 'IUPred3_final': 'Disorder (PTM-adj)'}
    )

    fig.update_traces(marker=dict(size=3.5, line=dict(color='rgba(30,30,30,0.35)', width=0.6)))
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig.show()

In [ ]:
#@title 6. Correlation: Protein Length vs. Hit Count
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Count hits per protein AND motif type
hits_by_type = df[df['type'] != 'No motifs found'].groupby(['protein_name', 'type']).size().reset_index(name='hit_count')

# Map lengths to these counts
hits_by_type['protein_length'] = hits_by_type['protein_name'].map(length_map)

# Create plot using consistent palette
plt.figure(figsize=(12, 7))
sns.lmplot(
    data=hits_by_type,
    x='protein_length',
    y='hit_count',
    hue='type',
    palette=MOTIF_COLORS,
    aspect=1.5,
    scatter_kws={'alpha':0.6}
)

plt.title('Correlation by Motif Type: Protein Length vs. Hit Count', fontsize=14)
plt.xlabel('Protein Length', fontsize=12)
plt.ylabel('Number of Motif Hits', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()